# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [12]:
print("Logistic Regression as my baseline since its readable and comparable to rule-based score.")
print("For main model, Random forest as a stronger variant of classifcation model as it captures non-linear relationships")
print("The reason it fits my lane: binary label")

Logistic Regression as my baseline since its readable and comparable to rule-based score.
For main model, Random forest as a stronger variant of classifcation model as it captures non-linear relationships
The reason it fits my lane: binary label


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [13]:
"""
Grouped by client, 80/20 since a random row split would let a client in training and in test so that would be leakage
"""

'\nGrouped by client, 80/20 since a random row split would let a client in training and in test so that would be leakage\n'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
df = pd.read_csv(
    "../../data/processed/refresh_feature_vector.csv"
)
numeric_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_cols = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

X_full = pd.get_dummies(df[numeric_cols + categorical_cols], columns=categorical_cols)
y_full = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_full, y_full, groups=df["client_id"]))

X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

assert set(df.iloc[train_idx]["client_id"]) & set(df_test["client_id"]) == set(), "client leakage in split!"

visible = df_test["impressions_90d"] >= 500
slip = (df_test["avg_position"] > 20) & (df_test["avg_position"] > 0)
stale = df_test["days_since_last_update"] >= 90

baseline_score_test = visible * (
    slip * df_test["impressions_90d"] + stale * df_test["impressions_90d"] * 0.3
)

log_reg = LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)

forest = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
forest.fit(X_train, y_train)

scores = {
    "Baseline": baseline_score_test.values,
    "LogisticRegression": log_reg.predict_proba(X_test_scaled)[:, 1],
    "RandomForest": forest.predict_proba(X_test)[:, 1],
}

def precision_at_k(y_true, score, k):
    k = min(k, len(y_true))
    top_k_idx = np.argsort(score)[::-1][:k]
    return y_true.values[top_k_idx].mean()

rows = []
base_rate = y_test.mean()

for name, score in scores.items():
    row = {
        "method": name,
        "precision@20": precision_at_k(y_test, score, 20),
        "precision@50": precision_at_k(y_test, score, 50),
        "precision@100": precision_at_k(y_test, score, 100),
        "average_precision": average_precision_score(y_test, score),
        "roc_auc": roc_auc_score(y_test, score),
    }
    if name != "Baseline":
        preds = (score >= 0.5).astype(int)
        row["precision"] = precision_score(y_test, preds)
        row["recall"] = recall_score(y_test, preds)
        row["f1"] = f1_score(y_test, preds)
        row["accuracy"] = accuracy_score(y_test, preds)
    rows.append(row)

results_table = pd.DataFrame(rows).set_index("method")
results_table["base_rate"] = base_rate
print(results_table.round(3))

                    precision@20  precision@50  precision@100  \
method                                                          
Baseline                    0.30          0.38           0.36   
LogisticRegression          0.70          0.72           0.71   
RandomForest                0.75          0.66           0.63   

                    average_precision  roc_auc  precision  recall     f1  \
method                                                                     
Baseline                        0.502    0.492        NaN     NaN    NaN   
LogisticRegression              0.604    0.616      0.576   0.721  0.641   
RandomForest                    0.602    0.608      0.582   0.647  0.613   

                    accuracy  base_rate  
method                                   
Baseline                 NaN      0.511  
LogisticRegression     0.587      0.511  
RandomForest           0.582      0.511  


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [15]:
rf_importance = pd.Series(forest.feature_importances_, index=X_train.columns)
rf_importance = rf_importance.sort_values(ascending=False)
print("Top 10 features (Random Forest):")
print(rf_importance.head(10))

logreg_coef = pd.Series(log_reg.coef_[0], index=X_train.columns)
logreg_coef = logreg_coef.sort_values(key=abs, ascending=False)
print("Top 10 features by |coefficient| (Logistic Regression):")
print(logreg_coef.head(10))

rf_proba = scores["RandomForest"]
df_test_scored = df_test.copy()
df_test_scored["y_true"] = y_test.values
df_test_scored["rf_proba"] = rf_proba

false_positives = df_test_scored[(df_test_scored["y_true"] == 0) & (df_test_scored["rf_proba"] > 0.7)]
false_negatives = df_test_scored[(df_test_scored["y_true"] == 1) & (df_test_scored["rf_proba"] < 0.3)]

print("False positives (model confident, actually not declining):")
print(false_positives[numeric_cols].head(2))

print("\nFalse negatives (model missed a real decline):")
print(false_negatives[numeric_cols].head(1))

Top 10 features (Random Forest):
log_impressions_90d      0.100757
avg_position             0.099355
days_with_impressions    0.082535
content_age_days         0.071239
word_count               0.057011
char_count               0.054715
log_sessions_90d         0.054111
days_with_sessions       0.051079
scroll_rate              0.050294
ctr                      0.049809
dtype: float64
Top 10 features by |coefficient| (Logistic Regression):
log_impressions_90d          1.454223
log_clicks_90d              -0.557561
word_count                   0.513762
avg_position                -0.378993
content_age_days            -0.348637
log_sessions_90d            -0.316077
impression_tier_low          0.304054
char_count                  -0.298076
word_count_tier_1000-2000    0.248978
position_tier_top_3         -0.232808
dtype: float64
False positives (model confident, actually not declining):
    search_volume  competition  cpc  word_count  char_count  \
13           10.0          0.0  0.0    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.